# 🎯 [MOV02_M03] 다익스트라 기반 최단거리 보급 경로 탐색 1차 Baseline 알고리즘 구현
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m03_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M03 |
| **미션 주제** | 다익스트라 기반 최단거리 보급 경로 탐색 1차 Baseline 알고리즘 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 다익스트라 기반 최단거리 보급 경로 탐색 1차 Baseline 알고리즘 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

다익스트라 기반 최단거리 보급 경로 탐색 Baseline 알고리즘을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **다익스트라 알고리즘으로 보급소에서 지원 부대까지 최단시간 경로를 찾는 Baseline과 경로 검증 규칙 엔진을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **간선 가중치를 차종별 소요시간으로 두고 통행 불가·하중 초과 구간을 제외해 최단 경로를 구 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/engine/mov02_rule_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 규정 및 지침 조항과의 정확한 일치성

#### 미션 PASS 조건

- 정량 기준: 업무 검증 룰 15종 이상 코딩, 룰 처리 지연시간 50ms 이하


- 진행 기준: 룰 엔진 단위 테스트 100% 통과


- 검증 방법: 경계 조건(Boundary condition)에 대한 룰 판정 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""다익스트라 알고리즘으로 보급소에서 지원 부대까지 최단시간 보급 경로를 찾는 Baseline을 구현하고, 설정 문자열로 분리한 경로 검증 규칙 15종과 경계 조건 단위 테스트로 결과를 판정한다."""
import heapq
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra as sp_dijkstra

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    # [TODO : 학생 실습 3] 다익스트라 기반 최단거리 보급 경로 탐색 1차 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def edge_minutes(e, asset="truck"):
    # [TODO : 학생 실습 3] 다익스트라 기반 최단거리 보급 경로 탐색 1차 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

# 규정·기준이 바뀌면 코드 대신 이 설정만 고친다(YAML 형식 문자열)
RULES_YAML = """
max_unpaved_ratio: 0.5
detour_factor: 2.5
max_threat_edges: 1
truck_count: 3
"""
CFG = {k.strip(): float(v) for k, v in (line.split(":") for line in RULES_YAML.strip().splitlines())}

nodes, edges = make_network()
threats = make_threats()
for e in edges:                                   # 미션 2 전처리 결과(폐쇄·하중 제한·통제 구역) 요약 반영
    if e["id"] in ("E017", "E040", "E046", "E071"):
        e["open"] = False
    if e["id"] in ("E025", "E064", "E083"):
        e.update(bridge=True, limit_t={"E025": 15.0, "E064": 20.0, "E083": 20.0}[e["id"]])
    if edge_risk(e, nodes, [{"x": 17.5, "y": 12.0, "r": 2.5, "p": 1}]) > 0:
        e["open"], e["zone"] = False, "CZ-1"
    e["risk"] = edge_risk(e, nodes, threats)
DEPOT = "N0000"
ASSET = {"kind": "truck", "status": "가용", **ASSETS["truck"]}


def route(dest, cargo_t, kind="truck"):
    gross = ASSETS[kind]["weight_t"] + min(cargo_t, ASSETS[kind]["cap_t"])
    adj = build_adj(nodes, edges, lambda e: edge_minutes(e, kind) if passable(e, gross) else None)
    return dijkstra(adj, DEPOT, dest)


def rules(req, cost, path, es, asset=ASSET):
    """경로 검증 규칙 15종. 결과: 규칙명 -> PASS / FAIL / WARN."""
    gross = asset["weight_t"] + asset["cap_t"]
    km = sum(e["km"] for e in es)
    straight = np.hypot(*np.subtract(nodes[req["dest"]], nodes[DEPOT]))
    pairs = {frozenset((e["u"], e["v"])) for e in edges}
    R = {
        "R01 출발지=보급소": path[:1] == [DEPOT],
        "R02 도착지=요청 부대": path[-1:] == [req["dest"]],
        "R03 연속 노드 간 도로 존재": all(frozenset(p) in pairs for p in zip(path, path[1:])),
        "R04 폐쇄 구간 미포함": all(e["open"] for e in es),
        "R05 교량 하중 이내": all(not e["bridge"] or e["limit_t"] >= gross for e in es),
        "R06 통제 구역 미통과": all("zone" not in e for e in es),
        "R07 노드 중복 방문 없음": len(path) == len(set(path)),
        "R08 거리 합계 정합": abs(km - sum(e["km"] for e in es)) < 1e-9 and len(es) == len(path) - 1,
        "R09 소요시간 기한 이내": cost <= req["deadline_h"] * 60,
        "R10 적재량 수송능력 이내": req["cargo_t"] <= asset["cap_t"] * CFG["truck_count"],
        "R11 자산 상태 가용": asset["status"] == "가용",
        "R12 비포장 비율 기준 이내": sum(e["km"] for e in es if e["road"] == "unpaved") <= CFG["max_unpaved_ratio"] * max(km, 1e-9),
        "R13 우회 과다 없음": km <= CFG["detour_factor"] * straight,
        "R14 경로 구간 1개 이상": len(es) >= 1,
        "R15 위협 구역 통과 기준 이내": sum(e["risk"] > 0 for e in es) <= CFG["max_threat_edges"],
    }
    out = {k: "PASS" if v else "FAIL" for k, v in R.items()}
    if out["R15 위협 구역 통과 기준 이내"] == "FAIL":
        out["R15 위협 구역 통과 기준 이내"] = "WARN"     # Baseline은 위협을 고려하지 않으므로 경고만 표시
    return out


rng = np.random.default_rng(2)
reqs = [{"req": f"REQ-{i:04d}", "dest": f"N{r:02d}07", "cargo_t": float(round(rng.uniform(4, 14), 1)), "deadline_h": 1.5} for i, r in enumerate(range(6))]
rows, lat, all_res = [], [], []
for q in reqs:
    cost, path, es = route(q["dest"], q["cargo_t"])
    t0 = time.perf_counter()
    res = rules(q, cost, path, es)
    lat.append((time.perf_counter() - t0) * 1000)
    all_res.append(res)
    rows.append({"요청": q["req"], "목적지": q["dest"], "거리(km)": round(sum(e["km"] for e in es), 1), "소요(분)": round(cost, 1),
                 "구간 수": len(es), "위협 통과 구간": sum(e["risk"] > 0 for e in es),
                 "PASS": sum(v == "PASS" for v in res.values()), "WARN": sum(v == "WARN" for v in res.values()), "FAIL": sum(v == "FAIL" for v in res.values())})
print("[Baseline 최단시간 경로 결과 - 중형 트럭]")
print(pd.DataFrame(rows).to_string(index=False))
print(f"\n검증 규칙 {len(all_res[0])}종(목표 15종 이상), 규칙 처리 평균 {np.mean(lat):.2f} ms / 최대 {np.max(lat):.2f} ms (목표 50 ms 이하: {'달성' if max(lat) <= 50 else '미달'})")

# 1) 최적성 검증: scipy 다익스트라와 비용 비교
ids = list(nodes); ix = {n: i for i, n in enumerate(ids)}
ok_e = [e for e in edges if passable(e, 10 + 5)]
w = [edge_minutes(e) for e in ok_e]
M = csr_matrix((w + w, ([ix[e["u"]] for e in ok_e] + [ix[e["v"]] for e in ok_e], [ix[e["v"]] for e in ok_e] + [ix[e["u"]] for e in ok_e])), shape=(len(ids),) * 2)
ref = sp_dijkstra(M, indices=ix[DEPOT])
same = sum(abs(ref[ix[q["dest"]]] - route(q["dest"], 5.0)[0]) < 1e-6 for q in reqs)
print(f"최적성 검증(scipy 결과와 일치): {same}/{len(reqs)}")

# 2) 경계 조건 단위 테스트
cost, path, es = route("N0507", 5.0)
tests = {
    "적재량=수송능력 한계(15t)": rules({"dest": "N0507", "cargo_t": 15.0, "deadline_h": 9}, cost, path, es)["R10 적재량 수송능력 이내"] == "PASS",
    "적재량 한계+0.1t": rules({"dest": "N0507", "cargo_t": 15.1, "deadline_h": 9}, cost, path, es)["R10 적재량 수송능력 이내"] == "FAIL",
    "소요시간=기한 정확히 일치": rules({"dest": "N0507", "cargo_t": 5, "deadline_h": cost / 60}, cost, path, es)["R09 소요시간 기한 이내"] == "PASS",
    "도달 불가 목적지 처리(고립 노드)": dijkstra(build_adj(nodes, edges, lambda e: None if "N0507" in (e["u"], e["v"]) else 1.0), DEPOT, "N0507") == (float("inf"), [], []),
    "출발지=도착지": dijkstra(build_adj(nodes, edges, lambda e: 1.0), DEPOT, DEPOT)[1] == [DEPOT],
    "정비 중 자산 차단": rules({"dest": "N0507", "cargo_t": 5, "deadline_h": 9}, cost, path, es, {**ASSET, "status": "정비"})["R11 자산 상태 가용"] == "FAIL",
}
for k, v in tests.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(f"단위 테스트 통과율 {np.mean(list(tests.values())):.0%}")
print("Baseline 한계: 시간만 최소화해 위협 구역을 지나는 경로(WARN)가 나온다 -> 미션 4에서 위협 회피·적재 효율을 반영한다.")

plt.figure(figsize=(8, 5.5))
for e in edges:
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    plt.plot([x1, x2], [y1, y2], color="lightgray" if e["open"] else "red", lw=1, ls="-" if e["open"] else "--")
for t in threats:
    plt.gca().add_patch(plt.Circle((t["x"], t["y"]), t["r"], color="red", alpha=0.12))
for q, c in zip(reqs, plt.cm.tab10.colors):
    _, path, _ = route(q["dest"], q["cargo_t"])
    xy = np.array([nodes[n] for n in path]); plt.plot(xy[:, 0], xy[:, 1], color=c, lw=2.5, label=f"to {q['dest']}")
plt.plot(*nodes[DEPOT], "ks", ms=10, label="depot")
plt.gca().set_aspect("equal"); plt.legend(fontsize=8, loc="upper left"); plt.xlabel("x (km)"); plt.ylabel("y (km)")
plt.title("Baseline shortest-time routes (red circles = threat zones, synthetic)")
plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[Baseline 최단시간 경로 결과 - 중형 트럭]
      요청   목적지  거리(km)  소요(분)  구간 수  위협 통과 구간  PASS  WARN  FAIL
REQ-0000 N0007    44.2   58.9     8         2    14     1     0
REQ-0001 N0107    40.2   54.1     7         2    14     1     0
REQ-0002 N0207    44.4   53.2     8         3    14     1     0
REQ-0003 N0307    49.0   58.8     9         3    14     1     0
REQ-0004 N0407    53.3   67.4    10         3    14     1     0
REQ-0005 N0507    58.4   73.5    11         3    14     1     0

검증 규칙 15종(목표 15종 이상), 규칙 처리 평균 0.06 ms / 최대 0.11 ms (목표 50 ms 이하: 달성)
최적성 검증(scipy 결과와 일치): 6/6
  [PASS] 적재량=수송능력 한계(15t)
  [PASS] 적재량 한계+0.1t
  [PASS] 소요시간=기한 정확히 일치
  [PASS] 도달 불가 목적지 처리(고립 노드)
  [PASS] 출발지=도착지
  [PASS] 정비 중 자산 차단
단위 테스트 통과율 100%
Baseline 한계: 시간만 최소화해 위협 구역을 지나는 경로(WARN)가 나온다 -> 미션 4에서 위협 회피·적재 효율을 반영한다.
```
</details>
